# 05 - Missing and Duplicate Data

Real extracts contain nulls, blank strings, inconsistent whitespace, and duplicate records.

## Learning objectives

By the end of this notebook, you will be able to:

- distinguish null from an empty string;
- standardise basic text values;
- use `isNull`, `coalesce`, `fillna`, and `dropna`; and
- remove exact duplicates safely.

## Prerequisite recap

Notebook 03 introduced column expressions and conditional logic. We will combine those tools into a multi-step cleaning transformation.

In [ ]:
from pyspark.sql import functions as F

customers = spark.createDataFrame(
    [
        ('C001', ' Northwind Supplies ', 'orders@northwind.example', None, 'North'),
        ('C002', 'Contoso Retail', None, 'contact@contoso.example', 'West'),
        ('C003', 'Adventure Works', None, None, '   '),
        ('C004', 'Fabrikam Stores', 'sales@fabrikam.example', None, 'South'),
        ('C004', 'Fabrikam Stores', 'sales@fabrikam.example', None, 'South'),
        (None, 'Unidentified Customer', None, 'unknown@example', None),
    ],
    'customer_id STRING, customer_name STRING, work_email STRING, personal_email STRING, region STRING',
)
customers.show(truncate=False)

## Null is not the same as blank

`NULL` means no value is present. `''` and whitespace such as `'   '` are still strings. Standardise blank strings to null before applying null-handling rules.

In [ ]:
customers_standardised = (
    customers
    .withColumn('customer_name', F.trim(F.col('customer_name')))
    .withColumn(
        'region',
        F.when(F.trim(F.col('region')) == '', F.lit(None))
        .otherwise(F.trim(F.col('region'))),
    )
)
customers_standardised.show(truncate=False)

## Find missing values

Use `isNull()` and `isNotNull()` inside a filter to inspect records before deciding how to handle them.

In [ ]:
customers_standardised.filter(F.col('work_email').isNull()).show()
customers_standardised.filter(F.col('customer_id').isNotNull()).show()

## Choose fallbacks, fill values, and reject unusable rows

- `coalesce` returns the first non-null expression.
- `fillna` supplies a known replacement.
- `dropna` removes rows missing required fields.

A missing customer ID is rejected because it cannot be joined reliably.

In [ ]:
customer_contacts = (
    customers_standardised
    .withColumn('preferred_email', F.coalesce('work_email', 'personal_email'))
    .dropna(subset=['customer_id'])
    .fillna({'region': 'Unknown'})
)
customer_contacts.show(truncate=False)

## Remove duplicates deliberately

`dropDuplicates()` removes rows that are identical across all columns. Supplying key columns, such as `dropDuplicates(['customer_id'])`, keeps an arbitrary record when those records differ. Notebook 10 shows how to choose the latest record deterministically.

In [ ]:
customers_clean = customer_contacts.dropDuplicates() # dropDuplicates(['customer_id']) for key specific checks
customers_clean.show(truncate=False)

print('Rows before exact deduplication:', customer_contacts.count())
print('Rows after exact deduplication:', customers_clean.count())

## A small data-quality check

Cleaning does not imply that every optional field is populated. Count remaining records without any usable email so the business can decide what to do.

In [16]:
missing_contact_count = customers_clean.filter(
    F.col('preferred_email').isNull()
).count()
print('Customers needing contact review:', missing_contact_count)

Customers needing contact review: 1


## Your turn

Use this separate store-contact dataset.

In [ ]:
store_contacts = spark.createDataFrame(
    [
        ('S001', '  Paris Central  ', 'paris@shops.example', None, '   '),
        ('S002', 'Lyon Outlet ', None, 'lyon.contact@shops.example', ' Lyon '),
        ('S003', ' Berlin Store', None, None, 'Berlin'),
        (None, 'Temporary Store', 'temp@shops.example', None, 'Nice'),
    ],
    'store_id STRING, store_name STRING, work_email STRING, backup_email STRING, city STRING',
)
store_contacts.show(truncate=False)

Create `stores_ready` from `store_contacts`:

- trim `store_name` and `city`;
- convert blank cities to null;
- create `preferred_email` from `work_email` and `backup_email`;
- remove rows without a `store_id`; and
- replace missing cities with `Unknown`.

In [ ]:
# Write your solution here.

### Expected result

`stores_ready` contains three rows. S001 has city `Unknown`; S002 uses its backup email as `preferred_email`; and S003 has a null `preferred_email`. Store names no longer have surrounding spaces.

### Solution - reveal after attempting

In [ ]:
stores_ready = (
    store_contacts
    .withColumn('store_name', F.trim(F.col('store_name')))
    .withColumn(
        'city',
        F.when(F.trim(F.col('city')) == '', F.lit(None))
        .otherwise(F.trim(F.col('city'))),
    )
    .withColumn('preferred_email', F.coalesce('work_email', 'backup_email'))
    .dropna(subset=['store_id'])
    .fillna({'city': 'Unknown'})
)
stores_ready.show(truncate=False)

## Key takeaway

Standardise first, apply explicit null rules, and deduplicate only when you understand which records may be discarded.

**Next:** change the grain of data with grouped calculations.